# Variable-Coefficient Drift-Diffusion and the Kolmogorov Equation

This tutorial demonstrates using **`symlie`** to analyze variable-coefficient diffusion models and Fokker–Planck–Kolmogorov equations, based on **F. Güngör** (*Lie symmetry group methods for differential equations*, arXiv:1901.01543):

1. **Variable-Coefficient Drift-Diffusion**: $u_t = x u_{xx} + b u_x$
   - Group classification over the parameter $b > 0$
   - Principal 4D finite essential symmetry algebra $\mathfrak{g}_4 = \langle \mathbf{v}_0, \mathbf{v}_1, \mathbf{v}_2, \mathbf{v}_3 \rangle$, in addition to solution superposition
   - Symmetry extension by two extra generators for the singular value $b = 1/2$
   - Equivalence mapping to the constant-coefficient heat equation $\tilde{u}_{\tilde{t}} = \tilde{u}_{\tilde{x}\tilde{x}}$
2. **The Generalized Fokker–Planck–Kolmogorov Equation**: $u_t - u_{xx} + x u_y = 0$
   - Multi-variable determining equations on $(t, x, y)$
   - Symmetries and group-invariant solutions

In [ ]:
import sympy as sp

from symlie import (
    InfinitesimalGenerator,
    determining_equations,
    max_derivative_order,
    verify_generator,
)

sp.init_printing()

x, t = sp.symbols("x t", positive=True)
b = sp.symbols("b", positive=True)
u = sp.Function("u")(x, t)

# Drift-diffusion equation: u_t = x*u_xx + b*u_x
drift_diff_eq = u.diff(t) - x * u.diff(x, 2) - b * u.diff(x)
print("PDE Order:", max_derivative_order(drift_diff_eq, u, (x, t)))
sp.Eq(drift_diff_eq, 0)

## 1. Symmetry Generators for General Parameter $b$

For arbitrary $b$, the equation admits a 4-dimensional finite essential symmetry algebra $\mathfrak{g}_4 = \langle \mathbf{v}_0, \mathbf{v}_1, \mathbf{v}_2, \mathbf{v}_3 \rangle$ (with $\mathbf{v}_0 = u\partial_u$ central inside this finite algebra) spanned by:
- $\mathbf{v}_1 = \partial_t$
- $\mathbf{v}_2 = t \partial_t + x \partial_x$
- $\mathbf{v}_3 = t^2 \partial_t + 2tx \partial_x - (x + bt)u \partial_u$
- $\mathbf{v}_0 = u \partial_u$

The complete algebra also contains $h(x,t)\partial_u$ for every solution $h$ of the homogeneous equation.

In [ ]:
v1 = InfinitesimalGenerator(xi=(0, 1), phi=(0,))
v2 = InfinitesimalGenerator(xi=(x, t), phi=(0,))
v3 = InfinitesimalGenerator(xi=(2 * t * x, t**2), phi=(-(x + b * t) * u,))
v0 = InfinitesimalGenerator(xi=(0, 0), phi=(u,))

for name, gen in [("v1", v1), ("v2", v2), ("v3", v3), ("v0", v0)]:
    valid = verify_generator(drift_diff_eq, u, (x, t), gen)
    print(f"Generator {name} invariant for general b: {valid}")

## 2. Symmetry Extension for the Singular Value $b = 1/2$

For $b = 1/2$, the finite essential algebra extends by two additional vector fields:
$$\mathbf{v}_4 = \sqrt{x}\partial_x, \quad \mathbf{v}_5 = t\sqrt{x}\partial_x - \sqrt{x} u \partial_u$$
forming a 6-dimensional finite essential algebra isomorphic to that of the linear heat equation $\tilde{u}_{\tilde{t}} = \tilde{u}_{\tilde{x}\tilde{x}}$. Both complete algebras additionally contain infinite solution-superposition ideals.

In [ ]:
eq_half = drift_diff_eq.subs(b, sp.Rational(1, 2))
v4_half = InfinitesimalGenerator(xi=(sp.sqrt(x), 0), phi=(0,))
v5_half = InfinitesimalGenerator(xi=(t * sp.sqrt(x), 0), phi=(-sp.sqrt(x) * u,))

print("b = 1/2, v4 invariant:", verify_generator(eq_half, u, (x, t), v4_half))
print("b = 1/2, v5 invariant:", verify_generator(eq_half, u, (x, t), v5_half))

# Equivalence transformation to standard heat equation: t_tilde = t, x_tilde = 2*sqrt(x)
u_heat = sp.Function("U")(2 * sp.sqrt(x), t)
sub_res = sp.simplify(drift_diff_eq.subs(b, sp.Rational(1, 2)).subs(u, u_heat).doit())
print("Transformed PDE in terms of U(2*sqrt(x), t):", sub_res)
print("Notice this simplifies precisely to the standard Heat equation U_t - U_XX!")

## 3. The Generalized Fokker–Planck–Kolmogorov Equation

We investigate the celebrated 3-variable Kolmogorov equation on $(t, x, y)$:
$$u_t - u_{xx} + x u_y = 0$$

We compute its determining equations using `determining_equations`.

In [ ]:
y = sp.symbols("y")
u_3d = sp.Function("u")(t, x, y)

kolmogorov_eq = u_3d.diff(t) - u_3d.diff(x, 2) + x * u_3d.diff(y)
print(
    "Kolmogorov Equation Order:", max_derivative_order(kolmogorov_eq, u_3d, (t, x, y))
)

det_kolm = determining_equations(kolmogorov_eq, u_3d, (t, x, y))
print(f"Determining equations derived: {len(det_kolm.equations)}")
for eq in det_kolm.equations[:5]:
    display(eq)

### Symmetry Verification of Kolmogorov Vector Fields

We verify the physical generators:
- Time translation: $\mathbf{v}_t = \partial_t$
- Spatial drift translation: $\mathbf{v}_y = \partial_y$
- Galilean boost: $\mathbf{v}_{\text{boost}} = \partial_x + t\partial_y$

In [ ]:
v_t = InfinitesimalGenerator(xi=(1, 0, 0), phi=(0,))
v_y = InfinitesimalGenerator(xi=(0, 0, 1), phi=(0,))
v_boost = InfinitesimalGenerator(xi=(0, 1, t), phi=(0,))
v_scale_u = InfinitesimalGenerator(xi=(0, 0, 0), phi=(u_3d,))

print("v_t invariant:      ", verify_generator(kolmogorov_eq, u_3d, (t, x, y), v_t))
print("v_y invariant:      ", verify_generator(kolmogorov_eq, u_3d, (t, x, y), v_y))
print("v_boost invariant:  ", verify_generator(kolmogorov_eq, u_3d, (t, x, y), v_boost))
print(
    "v_scale_u invariant:", verify_generator(kolmogorov_eq, u_3d, (t, x, y), v_scale_u)
)